# 🚀 Enterprise Voice RAG ChatBot — Google Colab GPU Backend
Run this notebook on Google Colab with **T4 GPU** runtime for sub-3 second voice response latency.

**Instructions**:
1. Go to **Runtime ➔ Change runtime type ➔ T4 GPU**.
2. Run **Cell 1** to clone/update the repository.
3. Run **Cell 2** to install GPU PyTorch & backend dependencies.
4. Run **Cell 3** to set your API keys and launch the backend server with ngrok.

In [ ]:
import os
%cd /content

if os.path.exists("/content/Voice_ChatBot/Voice_ChatBot"):
    !rm -rf /content/Voice_ChatBot/Voice_ChatBot

if not os.path.exists("/content/Voice_ChatBot"):
    !git clone https://github.com/FENGFANCHEN-012/Voice_ChatBot.git
    %cd /content/Voice_ChatBot
else:
    %cd /content/Voice_ChatBot
    !git pull

print("✅ Repository is ready at /content/Voice_ChatBot")

In [ ]:
%cd /content/Voice_ChatBot

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install -r backend/requirements.txt -q
!pip install pyngrok rank-bm25 llama-index-embeddings-huggingface -q

print("✅ All GPU dependencies installed successfully!")

In [ ]:
import os
from pyngrok import ngrok
from google.colab import userdata

# 1. Patch config.py
config_path = "/content/Voice_ChatBot/backend/app/config.py"
if os.path.exists(config_path):
    with open(config_path, "w", encoding="utf-8") as f:
        f.write('from pydantic_settings import BaseSettings\n\n\nclass Settings(BaseSettings):\n    gemini_api_key: str = ""\n    \n    # LLM Provider: "gemini" (default), "deepseek", or "auto"\n    llm_provider: str = "deepseek"\n    deepseek_api_key: str = ""\n    deepseek_base_url: str = "https://api.deepseek.com"\n    deepseek_model_name: str = "deepseek-chat"\n    \n    Qdrant_api_key: str = ""\n    \n    host: str = "0.0.0.0"\n    port: int = 8000\n    cors_origins: str = "http://localhost:5173"\n    whisper_model_size: str = "base"\n    whisper_use_gpu: bool = True\n    tts_voice: str = "en-US-AndrewMultilingualNeural"\n    tts_rate: str = "+0%"\n    tts_pitch: str = "+0Hz"\n    \n    # ------------------------------------------\n    \n    # fast embedding model\n    embedding_model_name: str = "BAAI/bge-m3"\n    \n    \n    reranker_model_name: str = "BAAI/bge-reranker-v2-m3"\n\n    # Vector store — "chroma" (default) or "faiss"\n    vector_store_type: str = "chroma"\n    chroma_db_path: str = "chroma_db"\n    faiss_index_path: str = "faiss_index/index.faiss"\n    faiss_metadata_path: str = "faiss_index/metadata.pkl"\n\n    upload_dir: str = "uploads"\n    max_file_size_mb: int = 40\n    chunk_size: int = 800\n    chunk_overlap: int = 150\n    retrieval_top_k: int = 30\n    retrieval_fetch_k: int = 40\n    reranker_top_k: int = 15\n\n\n\n    # Advanced retrieval (AgentRAG classification + QueryExpander). Only used\n    # when running on GPU (CUDA available). Set to false to disable.\n    use_advanced_pipeline: bool = True\n\n\n    model_config = {"env_file": ".env", "env_file_encoding": "utf-8", "extra": "ignore"}\n\n\nsettings = Settings()\n')
    print("✅ Verified and patched config.py")

# 2. Patch llm.py
llm_path = "/content/Voice_ChatBot/backend/app/pipeline/llm.py"
if os.path.exists(llm_path):
    with open(llm_path, "w", encoding="utf-8") as f:
        f.write('import asyncio\nfrom loguru import logger\nimport google.generativeai as genai\nfrom app.pipeline.rate_limiter import gemini_rate_limiter\n\n\nSTATIC_PROMPT = """You are an enterprise policy assistant. Answer the user\'s question based ONLY on the provided context and conversation history.\n\nRules:\n1. Cite specific references from the context: exact chapter numbers, error codes (e.g. ERR-SSO-4039), form numbers (e.g. Form HR-PAY-102), directive names, and policy codes.\n2. Provide complete, detailed, and exhaustive factual answers. Include all specific time limits, business days, thresholds, temperature metrics, and department roles mentioned in the context.\n3. When a cross-domain dependency exists (e.g. "See Chapter 3"), mention it explicitly.\n4. For comparison questions (e.g., comparing two error codes, procedures, or policies), compare and contrast both requested items explicitly, detailing the exact rules, forms, timeframes, and actions for each.\n5. For step-by-step procedures, list all steps clearly in sequence.\n6. If the question is a follow-up from conversation history (e.g. "explain more", "what do you mean"), answer using history context.\n7. If the question clearly requires document context that isn\'t available in the provided text, state: "I cannot find this information in the uploaded documents."\n8. Do NOT hallucinate or make up information not present in the context.\n\nBelow is the conversation history, document context, and the question."""\n\n\n\n\nimport json\nimport httpx\nfrom app.config import settings\n\n\nclass LLMClient:\n    def __init__(self, api_key: str):\n        if api_key:\n            genai.configure(api_key=api_key)\n        self.fallback_models = ["gemini-2.0-flash-lite", "gemini-1.5-flash", "gemini-2.0-flash"]\n        self.model_name = self.fallback_models[0]\n        self.last_rate_limit_wait: float = 0.0\n\n    def _build_dynamic_prompt(self, query: str, context: list[str], history: list[dict] | None = None) -> str:\n        docs = "\\n\\n".join(f"---\\n{c}" for c in context)\n\n        history_block = ""\n        if history:\n            lines = []\n            for msg in history[-6:]:\n                role = "User" if msg["role"] == "user" else "Assistant"\n                lines.append(f"{role}: {msg[\'content\']}")\n            history_block = "\\n".join(lines) + "\\n\\n"\n\n        return f"""Conversation History:\n{history_block}\nContext:\n{docs}\n\nQuestion: {query}"""\n\n    async def _generate_deepseek(self, dynamic: str) -> str:\n        api_key = settings.deepseek_api_key or settings.gemini_api_key\n        url = f"{settings.deepseek_base_url.rstrip(\'/\')}/chat/completions"\n        headers = {\n            "Authorization": f"Bearer {api_key}",\n            "Content-Type": "application/json"\n        }\n        payload = {\n            "model": settings.deepseek_model_name,\n            "messages": [\n                {"role": "system", "content": STATIC_PROMPT},\n                {"role": "user", "content": dynamic}\n            ],\n            "temperature": 0.3\n        }\n        async with httpx.AsyncClient(timeout=60.0) as client:\n            resp = await client.post(url, headers=headers, json=payload)\n            resp.raise_for_status()\n            data = resp.json()\n            return data["choices"][0]["message"]["content"]\n\n    async def _generate_deepseek_stream(self, dynamic: str):\n        api_key = settings.deepseek_api_key or settings.gemini_api_key\n        url = f"{settings.deepseek_base_url.rstrip(\'/\')}/chat/completions"\n        headers = {\n            "Authorization": f"Bearer {api_key}",\n            "Content-Type": "application/json"\n        }\n        payload = {\n            "model": settings.deepseek_model_name,\n            "messages": [\n                {"role": "system", "content": STATIC_PROMPT},\n                {"role": "user", "content": dynamic}\n            ],\n            "stream": True,\n            "temperature": 0.3\n        }\n        async with httpx.AsyncClient(timeout=60.0) as client:\n            async with client.stream("POST", url, headers=headers, json=payload) as resp:\n                resp.raise_for_status()\n                async for line in resp.aiter_lines():\n                    if line.startswith("data: "):\n                        data_str = line[6:].strip()\n                        if data_str == "[DONE]":\n                            break\n                        try:\n                            data = json.loads(data_str)\n                            delta = data["choices"][0]["delta"].get("content", "")\n                            if delta:\n                                yield delta\n                        except Exception:\n                            continue\n\n    async def generate(self, query: str, context: list[str], history: list[dict] | None = None) -> str:\n        dynamic = self._build_dynamic_prompt(query, context, history)\n        self.last_rate_limit_wait = 0.0\n\n        provider = settings.llm_provider.lower()\n        if provider == "deepseek" or (provider == "auto" and settings.deepseek_api_key):\n            try:\n                logger.info("[LLM] Generating answer using DeepSeek V3...")\n                return await self._generate_deepseek(dynamic)\n            except Exception as e:\n                logger.error(f"[LLM] DeepSeek failed ({e})")\n                if not settings.gemini_api_key:\n                    return f"DeepSeek API Error: {e}"\n                logger.warning("[LLM] Falling back to Gemini...")\n\n        if not settings.gemini_api_key:\n            return "Error: No valid LLM API key configured (neither DeepSeek nor Gemini)."\n\n        for model_name in self.fallback_models:\n            model = genai.GenerativeModel(model_name, system_instruction=STATIC_PROMPT)\n            for attempt in range(2):\n                try:\n                    waited = await gemini_rate_limiter.acquire()\n                    self.last_rate_limit_wait += waited\n                    response = await asyncio.to_thread(model.generate_content, dynamic)\n                    return response.text\n                except Exception as e:\n                    if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):\n                        logger.warning(f"[LLM] Quota hit on {model_name}, trying fallback model...")\n                        break\n                    else:\n                        raise\n        logger.error("[LLM] Quota exceeded on all fallback models")\n        return "I\'m experiencing high demand. Please try again in a minute."\n\n    async def generate_stream(self, query: str, context: list[str], history: list[dict] | None = None):\n        dynamic = self._build_dynamic_prompt(query, context, history)\n\n        provider = settings.llm_provider.lower()\n        if provider == "deepseek" or (provider == "auto" and settings.deepseek_api_key):\n            try:\n                logger.info("[LLM] Streaming answer using DeepSeek V3...")\n                async for chunk in self._generate_deepseek_stream(dynamic):\n                    yield chunk\n                return\n            except Exception as e:\n                logger.error(f"[LLM] DeepSeek streaming failed ({e})")\n                if not settings.gemini_api_key:\n                    yield f"DeepSeek API Error: {e}"\n                    return\n                logger.warning("[LLM] Falling back to Gemini...")\n\n        if not settings.gemini_api_key:\n            yield "Error: No valid LLM API key configured (neither DeepSeek nor Gemini)."\n            return\n\n        for model_name in self.fallback_models:\n            model = genai.GenerativeModel(model_name, system_instruction=STATIC_PROMPT)\n            for attempt in range(2):\n                try:\n                    await gemini_rate_limiter.acquire()\n                    response = await asyncio.to_thread(model.generate_content, dynamic, stream=True)\n                    for chunk in response:\n                        if chunk.text:\n                            yield chunk.text\n                    return\n                except Exception as e:\n                    if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):\n                        logger.warning(f"[LLM] Quota hit on {model_name}, trying fallback model...")\n                        break\n                    else:\n                        raise\n        logger.error("[LLM] Quota exceeded on all fallback models")\n        yield "I\'m experiencing high demand. Please try again in a minute."\n')
    print("✅ Verified and patched llm.py")

# 3. Patch audio_service.py
audio_path = "/content/Voice_ChatBot/backend/app/services/audio_service.py"
if os.path.exists(audio_path):
    with open(audio_path, "w", encoding="utf-8") as f:
        f.write('import asyncio\nimport io\nimport os\nimport re\nimport hashlib\nimport tempfile\nfrom pathlib import Path\n\nimport edge_tts\nfrom faster_whisper import WhisperModel\nfrom app.config import settings\n\n_ffmpeg_dirs = [\n    str(Path(__file__).resolve().parent.parent.parent.parent / "venv" / "Lib" / "site-packages" / "imageio_ffmpeg" / "binaries"),\n    str(Path(__file__).resolve().parent.parent.parent.parent / ".venv" / "Lib" / "site-packages" / "imageio_ffmpeg" / "binaries"),\n]\nfor d in _ffmpeg_dirs:\n    if os.path.isdir(d):\n        os.environ["PATH"] = d + os.pathsep + os.environ.get("PATH", "")\n        break\n\n\ndef clean_text_for_tts(text: str) -> str:\n    # Replace slashes between word characters or standalone slashes with spaces so TTS doesn\'t say "slash"\n    text = re.sub(r\'(?<=\\w)/(?=\\w)\', \' \', text)\n    text = re.sub(r\'[/\\\\#*_`~|{}]\', \' \', text)\n    text = re.sub(r\'[—–]\', \', \', text)\n    text = re.sub(r\'\\s+-\\s+\', \', \', text)\n    text = re.sub(r\'(?m)^\\s*[-•]\\s+\', \'\', text)\n    text = re.sub(r\'\\s+\', \' \', text).strip()\n    return text\n\n\nclass TTSCache:\n    def __init__(self, max_size: int = 100):\n        self._cache: dict[str, bytes] = {}\n        self._max_size = max_size\n\n    def _key(self, text: str) -> str:\n        return hashlib.md5(text.encode()).hexdigest()\n\n    def get(self, text: str) -> bytes | None:\n        return self._cache.get(self._key(text))\n\n    def set(self, text: str, audio: bytes):\n        if len(self._cache) >= self._max_size:\n            oldest = next(iter(self._cache))\n            del self._cache[oldest]\n        self._cache[self._key(text)] = audio\n\n    def clear(self):\n        self._cache.clear()\n\n\nclass AudioService:\n    def __init__(self, model_size: str = "base"):\n        use_gpu = settings.whisper_use_gpu\n        device = "cuda" if use_gpu else "cpu"\n        compute = "float16" if use_gpu else "int8"\n        try:\n            import torch\n            if use_gpu and not torch.cuda.is_available():\n                device = "cpu"\n                compute = "int8"\n        except ImportError:\n            device = "cpu"\n            compute = "int8"\n        self.whisper = WhisperModel(model_size, device=device, compute_type=compute)\n        self.tts_cache = TTSCache(max_size=100)\n\n    async def transcribe(self, audio_data: bytes, filename: str = "audio.webm") -> dict:\n        suffix = Path(filename).suffix or ".webm"\n\n        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:\n            tmp.write(audio_data)\n            tmp_path = tmp.name\n\n        try:\n            def _do_transcribe():\n                segments, info = self.whisper.transcribe(tmp_path, beam_size=1)\n                text = " ".join(seg.text for seg in segments)\n                return text.strip(), info.language, info.duration\n\n            text, lang, duration = await asyncio.to_thread(_do_transcribe)\n            return {"text": text, "language": lang, "duration": duration}\n        finally:\n            Path(tmp_path).unlink(missing_ok=True)\n\n    async def synthesize(self, text: str) -> bytes:\n        text = clean_text_for_tts(text[:500])\n        if not text:\n            return b""\n\n        cached = self.tts_cache.get(text)\n        if cached:\n            return cached\n\n        voices_to_try = [settings.tts_voice, "en-US-AvaNeural", "en-US-ChristopherNeural"]\n        for v in voices_to_try:\n            try:\n                async def _stream_tts(voice_name: str):\n                    communicate = edge_tts.Communicate(text, voice=voice_name, rate=settings.tts_rate, pitch=settings.tts_pitch)\n                    buf = io.BytesIO()\n                    async for chunk in communicate.stream():\n                        if chunk["type"] == "audio":\n                            buf.write(chunk["data"])\n                    buf.seek(0)\n                    return buf.getvalue()\n\n                audio = await asyncio.wait_for(_stream_tts(v), timeout=12.0)\n                if audio and len(audio) > 100:\n                    self.tts_cache.set(text, audio)\n                    return audio\n            except Exception as e:\n                from loguru import logger\n                logger.warning(f"[TTS] Edge-TTS voice {v} failed or timed out: {e}")\n                continue\n\n        return b""\n\n\n    async def synthesize_stream(self, text: str):\n        text = clean_text_for_tts(text[:500])\n        if not text:\n            return\n\n        cached = self.tts_cache.get(text)\n        if cached:\n            yield cached\n            return\n\n        try:\n            communicate = edge_tts.Communicate(text, voice=settings.tts_voice, rate=settings.tts_rate, pitch=settings.tts_pitch)\n            buf = io.BytesIO()\n            async for chunk in communicate.stream():\n                if chunk["type"] == "audio":\n                    buf.write(chunk["data"])\n                    yield chunk["data"]\n            buf.seek(0)\n            audio = buf.getvalue()\n            if audio:\n                self.tts_cache.set(text, audio)\n        except Exception as e:\n            from loguru import logger\n            logger.warning(f"[TTS] Edge-TTS stream failed: {e}")\n')
    print("✅ Verified and patched audio_service.py")

# 4. Patch hybrid_search.py
hybrid_path = "/content/Voice_ChatBot/backend/app/pipeline/hybrid_search.py"
if os.path.exists(hybrid_path):
    with open(hybrid_path, "w", encoding="utf-8") as f:
        f.write('import numpy as np\nfrom loguru import logger\nfrom rank_bm25 import BM25Okapi\nfrom app.pipeline.query_normalizer import normalize_query, extract_metadata_hints\n\n\nclass HybridSearch:\n    def __init__(self, vector_store, embedder, k: int = 20, rrf_k: int = 60):\n        self.vector_store = vector_store\n        self.embedder = embedder\n        self.k = k\n        self.rrf_k = rrf_k\n        self._bm25 = None\n        self._bm25_texts = []\n        self._bm25_metadata = []\n\n    def rebuild(self):\n        all_chunks = self.vector_store.get_all_chunks_with_metadata()\n        if not all_chunks:\n            self._bm25 = None\n            self._bm25_texts = []\n            self._bm25_metadata = []\n            return\n        self._bm25_texts = [c["text"] for c in all_chunks]\n        self._bm25_metadata = [c.get("metadata", {}) for c in all_chunks]\n        tokenized = [doc.lower().split() for doc in self._bm25_texts]\n        self._bm25 = BM25Okapi(tokenized)\n\n    def _decompose_query(self, query_text: str) -> list[str]:\n        q_lower = query_text.lower()\n        sub_queries = [query_text]\n\n        # Decompose comparison / multi-entity queries\n        if "between " in q_lower and " and " in q_lower:\n            try:\n                after_between = q_lower.split("between ", 1)[1]\n                parts = after_between.split(" and ", 1)\n                if len(parts) == 2:\n                    clean_a = parts[0].replace("?", "").strip()\n                    clean_b = parts[1].replace("?", "").strip()\n                    sub_queries.append(f"{query_text} {clean_a}")\n                    sub_queries.append(f"{query_text} {clean_b}")\n            except Exception:\n                pass\n        elif " differ" in q_lower or " difference" in q_lower or " compare" in q_lower:\n            tokens = [t.strip() for t in query_text.replace("?", "").split(" and ") if len(t.strip()) > 3]\n            sub_queries.extend(tokens)\n\n        # Remove duplicates while preserving order\n        seen = set()\n        unique_queries = []\n        for q in sub_queries:\n            if q not in seen:\n                seen.add(q)\n                unique_queries.append(q)\n\n        return unique_queries[:3]\n\n    def _single_search(self, query_text: str, where: dict | None = None) -> list[dict]:\n        if self._bm25 is None:\n            self.rebuild()\n\n        normalized_query = normalize_query(query_text)\n        hints = extract_metadata_hints(normalized_query)\n\n        if hints.get("chapter") and where is None:\n            where = {"chapter": hints["chapter"]}\n\n        query_vector = self.embedder.embed_query(normalized_query)\n        vec_results = self.vector_store.search(query_vector, k=self.k, where=where)\n\n        filter_used = where\n        if not vec_results and where:\n            vec_results = self.vector_store.search(query_vector, k=self.k, where=None)\n            filter_used = None\n\n        tokenized_query = normalized_query.lower().split()\n        bm25_scores = self._bm25.get_scores(tokenized_query) if self._bm25 else []\n\n        bm25_ranked = []\n        for i, score in enumerate(bm25_scores):\n            if score > 0:\n                if filter_used and self._bm25_metadata:\n                    meta = self._bm25_metadata[i] if i < len(self._bm25_metadata) else {}\n                    if filter_used.get("chapter") and meta.get("chapter") != filter_used["chapter"]:\n                        continue\n                bm25_ranked.append((i, score))\n\n        bm25_ranked = sorted(bm25_ranked, key=lambda x: x[1], reverse=True)[:self.k]\n\n        fused = {}\n        for idx, result in enumerate(vec_results):\n            text = result["text"]\n            fused[text] = {\n                "vec_rank": idx,\n                "result": result,\n            }\n\n        for idx, (doc_idx, _) in enumerate(bm25_ranked):\n            text = self._bm25_texts[doc_idx]\n            if text in fused:\n                fused[text]["bm25_rank"] = idx\n            else:\n                fused[text] = {\n                    "vec_rank": None,\n                    "bm25_rank": idx,\n                    "result": {"text": text, "score": 0.0},\n                }\n\n        for text, data in fused.items():\n            vec_score = 1.0 / (self.rrf_k + data["vec_rank"] + 1) if data["vec_rank"] is not None else 0\n            bm25_score = 1.0 / (self.rrf_k + data["bm25_rank"] + 1) if "bm25_rank" in data else 0\n            data["rrf_score"] = vec_score + bm25_score\n\n        ranked = sorted(fused.values(), key=lambda x: x["rrf_score"], reverse=True)[:self.k]\n        results = []\n        for item in ranked:\n            r = dict(item["result"])\n            r["score"] = float(item["rrf_score"])\n            results.append(r)\n\n        return results\n\n    def search(self, query_text: str, where: dict | None = None) -> list[dict]:\n        sub_queries = self._decompose_query(query_text)\n        \n        all_results_map = {}\n        for q in sub_queries:\n            sub_res = self._single_search(q, where=where)\n            for r in sub_res:\n                txt = r["text"]\n                if txt not in all_results_map or r["score"] > all_results_map[txt]["score"]:\n                    all_results_map[txt] = r\n\n        combined = sorted(all_results_map.values(), key=lambda x: x["score"], reverse=True)[:self.k]\n        logger.info(f"[HybridSearch] Multi-query search ({len(sub_queries)} queries) returned {len(combined)} chunks")\n        return combined\n\n')
    print("✅ Verified and patched hybrid_search.py")

# 5. Patch orchestrator.py
orch_path = "/content/Voice_ChatBot/backend/app/pipeline/orchestrator.py"
if os.path.exists(orch_path):
    with open(orch_path, "w", encoding="utf-8") as f:
        f.write('import asyncio\nimport torch\nfrom loguru import logger\nfrom app.config import settings\nfrom app.pipeline.retriever import Retriever\nfrom app.pipeline.reranker import Reranker\nfrom app.pipeline.llm import LLMClient\nfrom app.pipeline.query_expander import QueryExpander\nfrom app.pipeline.hybrid_search import HybridSearch\nfrom app.pipeline.fallback import Fallback\nfrom app.pipeline.agent_rag import AgentRAG\nfrom app.pipeline.query_cache import QueryCache\nfrom app.pipeline.hyde import HyDE\n\n\nclass PipelineOrchestrator:\n    def __init__(\n        self,\n        retriever: Retriever,\n        reranker: Reranker,\n        llm: LLMClient,\n        query_expander: QueryExpander,\n        hybrid: HybridSearch,\n        fallback: Fallback,\n        agent: AgentRAG,\n        hyde: HyDE | None = None,\n    ):\n        self.retriever = retriever\n        self.reranker = reranker\n        self.llm = llm\n        self.query_expander = query_expander\n        self.hybrid = hybrid\n        self.fallback = fallback\n        self.agent = agent\n        self.hyde = hyde\n        self.cache = QueryCache(ttl_seconds=3600, max_size=100)\n        self.use_advanced = settings.use_advanced_pipeline and torch.cuda.is_available()\n        logger.info(f"[Pipeline] Advanced retrieval (AgentRAG + QueryExpander): "\n                    f"{\'ENABLED\' if self.use_advanced else \'disabled (no CUDA GPU)\'}")\n\n    async def answer_question(self, text: str, history: list[dict] | None = None) -> dict:\n        logger.info(f"[Pipeline] Query: {text}")\n\n        cached = self.cache.get(text)\n        if cached:\n            logger.info(f"[Pipeline] Cache hit (cache size: {self.cache.size})")\n            return cached\n\n        logger.info("[Pipeline] Cache miss → simple path (no agent, no expansion)")\n        result = await self._simple_path(text, history)\n        self.cache.set(text, result)\n        return result\n\n    async def answer_question_stream(self, text: str, history: list[dict] | None = None):\n        logger.info(f"[Pipeline] Stream query: {text}")\n\n        cached = self.cache.get(text)\n        if cached:\n            logger.info(f"[Pipeline] Cache hit (cache size: {self.cache.size})")\n            yield {"type": "token", "content": cached["answer_text"]}\n            yield {"type": "done", "chunks": cached["chunks"]}\n            return\n\n        logger.info("[Pipeline] Cache miss → streaming path")\n\n        category = None\n        if self.use_advanced:\n            category, chunks = await self._advanced_retrieve(text)\n            if category == self.agent.OUT_OF_SCOPE:\n                logger.info("[Pipeline] Query classified out-of-scope; answering without context")\n                full_answer = ""\n                async for token in self.llm.generate_stream(text, [], history=history):\n                    full_answer += token\n                    yield {"type": "token", "content": token}\n                self.cache.set(text, {"answer_text": full_answer, "chunks": []})\n                yield {"type": "done", "chunks": []}\n                return\n        else:\n            chunks = await asyncio.to_thread(self.hybrid.search, text)\n\n        top_chunks = await asyncio.to_thread(self.reranker.rerank, text, chunks, top_k=settings.reranker_top_k)\n        context = [c["text"] for c in top_chunks]\n\n        full_answer = ""\n        async for token in self.llm.generate_stream(text, context, history=history):\n            full_answer += token\n            yield {"type": "token", "content": token}\n\n        result_chunks = [\n            {"content": c["text"], "page": c.get("page"), "score": float(c.get("score", 0))}\n            for c in top_chunks\n        ]\n\n        self.cache.set(text, {"answer_text": full_answer, "chunks": result_chunks})\n        yield {"type": "done", "chunks": result_chunks}\n\n    async def _handle_out_of_scope(self, text: str, history: list[dict] | None = None) -> dict:\n        answer = await self.llm.generate(text, [], history=history)\n        return {\n            "answer_text": answer,\n            "rate_limit_wait": getattr(self.llm, "last_rate_limit_wait", 0.0),\n            "chunks": [],\n        }\n\n    async def _simple_path(self, original: str, history: list[dict] | None = None) -> dict:\n        category = None\n        if self.use_advanced:\n            category, chunks = await self._advanced_retrieve(original)\n            if category == self.agent.OUT_OF_SCOPE:\n                logger.info("[Pipeline] Query classified out-of-scope; answering without context")\n                answer = await self.llm.generate(original, [], history=history)\n                return {\n                    "answer_text": answer,\n                    "rate_limit_wait": getattr(self.llm, "last_rate_limit_wait", 0.0),\n                    "chunks": [],\n                }\n        else:\n            chunks = await asyncio.to_thread(self.hybrid.search, original)\n\n        top_chunks = await asyncio.to_thread(self.reranker.rerank, original, chunks, top_k=settings.reranker_top_k)\n        context = [c["text"] for c in top_chunks]\n        answer = await self.llm.generate(original, context, history=history)\n        return {\n            "answer_text": answer,\n            "rate_limit_wait": getattr(self.llm, "last_rate_limit_wait", 0.0),\n            "chunks": [\n                {"content": c["text"], "page": c.get("page"), "score": float(c.get("score", 0))}\n                for c in top_chunks\n            ],\n        }\n\n    async def _advanced_retrieve(self, original: str) -> tuple[str, list[dict]]:\n        """Classify the query with AgentRAG, expand it with QueryExpander, then\n        run multi-query hybrid search and merge the results. Any failure degrades\n        gracefully to plain hybrid search."""\n        category = None\n        try:\n            category = await self.agent.classify(original)\n            logger.info(f"[Pipeline] AgentRAG classified query as: {category}")\n        except Exception as e:\n            logger.warning(f"[Pipeline] AgentRAG classify failed ({e}); using plain hybrid search")\n\n        if category == self.agent.OUT_OF_SCOPE:\n            return category, []\n\n        expanded = original\n        try:\n            expanded = await self.query_expander.expand(original)\n            logger.info(f"[Pipeline] QueryExpander: \'{original}\' -> \'{expanded}\'")\n        except Exception as e:\n            logger.warning(f"[Pipeline] QueryExpander failed ({e}); using original query")\n\n        sub_queries = self._build_sub_queries(original, expanded)\n\n        all_results: dict[str, dict] = {}\n        for q in sub_queries:\n            for r in await asyncio.to_thread(self.hybrid.search, q):\n                key = r["text"]\n                if key not in all_results or r["score"] > all_results[key]["score"]:\n                    all_results[key] = r\n\n        chunks = sorted(all_results.values(), key=lambda x: x["score"], reverse=True)\n        chunks = chunks[: settings.retrieval_fetch_k]\n        logger.info(f"[Pipeline] Advanced retrieval: {len(sub_queries)} sub-queries → {len(chunks)} chunks")\n        return category or self.agent.COMPLEX, chunks\n\n    def _build_sub_queries(self, original: str, expanded: str) -> list[str]:\n        queries = [original]\n        if expanded and expanded != original:\n            parts = [p.strip() for p in expanded.split(",") if p.strip()]\n            queries.extend(parts[:4])\n        seen = set()\n        unique = []\n        for q in queries:\n            if q and q not in seen:\n                seen.add(q)\n                unique.append(q)\n        return unique[:5]\n\n    async def _full_path(self, original: str, history: list[dict] | None = None) -> dict:\n        result = await self.fallback.execute(original, history=history)\n\n        if result["passed"]:\n            final_answer = result["answer_text"]\n            chunks = result["chunks"]\n            rate_limit_wait = getattr(self.llm, "last_rate_limit_wait", 0.0)\n        else:\n            chunks = await asyncio.to_thread(self.hybrid.search, original)\n            top_chunks = await asyncio.to_thread(self.reranker.rerank, original, chunks, top_k=settings.reranker_top_k)\n            context = [c["text"] for c in top_chunks]\n            final_answer = await self.llm.generate(original, context, history=history)\n            rate_limit_wait = getattr(self.llm, "last_rate_limit_wait", 0.0)\n            chunks = top_chunks\n\n        return {\n            "answer_text": final_answer,\n            "rate_limit_wait": rate_limit_wait,\n            "chunks": [\n                {"content": c["text"], "page": c.get("page"), "score": float(c.get("score", 0))}\n                for c in chunks\n            ],\n        }\n')
    print("✅ Verified and patched orchestrator.py")

deepseek_key = ""
try:
    deepseek_key = userdata.get("DEEPSEEK_API_KEY")
except Exception:
    pass
if not deepseek_key:
    deepseek_key = os.environ.get("DEEPSEEK_API_KEY", "").strip()
if not deepseek_key:
    deepseek_key = input("Enter DEEPSEEK_API_KEY (or press Enter for Gemini): ").strip()

gemini_key = ""
try:
    gemini_key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass
if not gemini_key:
    gemini_key = os.environ.get("GEMINI_API_KEY", "").strip()
if not gemini_key and not deepseek_key:
    gemini_key = input("Enter GEMINI_API_KEY: ").strip()

if deepseek_key:
    os.environ["DEEPSEEK_API_KEY"] = deepseek_key
if gemini_key:
    os.environ["GEMINI_API_KEY"] = gemini_key

provider = "deepseek" if deepseek_key else "gemini"
os.environ["LLM_PROVIDER"] = provider

env_path = "/content/Voice_ChatBot/backend/.env"
os.makedirs(os.path.dirname(env_path), exist_ok=True)
with open(env_path, "w", encoding="utf-8") as f:
    f.write(f"LLM_PROVIDER={provider}\n")
    if deepseek_key:
        f.write(f"DEEPSEEK_API_KEY={deepseek_key}\n")
        f.write("DEEPSEEK_BASE_URL=https://api.deepseek.com\n")
        f.write("DEEPSEEK_MODEL_NAME=deepseek-chat\n")
    if gemini_key:
        f.write(f"GEMINI_API_KEY={gemini_key}\n")
    f.write("VECTOR_STORE_TYPE=chroma\n")

!pkill -f uvicorn
!pkill -f ngrok
ngrok.kill()

ngrok_token = input("Enter NGROK_AUTHTOKEN (or press Enter if configured): ").strip()
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

try:
    public_url = ngrok.connect(8000).public_url
    print("\n" + "=" * 70)
    print("🚀 CLOUD GPU BACKEND IS LIVE!")
    print(f"👉 NGROK PUBLIC URL: {public_url}")
    print(f"🤖 ACTIVE LLM PROVIDER: {provider.upper()}")
    print("📋 Copy this URL and set VITE_BACKEND_URL in frontend/.env")
    print("=" * 70 + "\n")
except Exception as e:
    print(f"⚠️ ngrok status: {e}")

%cd /content/Voice_ChatBot/backend
!python -m uvicorn app.main:app --host 0.0.0.0 --port 8000